# 02. Data Preparation

## Objective

This notebook prepares the FAA Wildlife Strike dataset for machine learning.

The preparation process includes:

- standardizing missing values;
- correcting data types;
- handling duplicate records;
- identifying and removing target leakage variables;
- selecting modelling features;
- exporting a clean dataset for the modelling stage.

## 1. Import Libraries

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

## 2. Define Project Paths

In [2]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "faa_strikes.csv"

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw dataset: {RAW_DATA_PATH}")
print(f"Processed folder: {PROCESSED_DATA_DIR}")

Project root: d:\Phuong\UNF\summer 2026\Capstone\faa-wildlife-strike-damage-analysis
Raw dataset: d:\Phuong\UNF\summer 2026\Capstone\faa-wildlife-strike-damage-analysis\data\raw\faa_strikes.csv
Processed folder: d:\Phuong\UNF\summer 2026\Capstone\faa-wildlife-strike-damage-analysis\data\processed


## 3. Load Dataset

In [3]:
df = pd.read_csv(
    RAW_DATA_PATH,
    encoding="latin1",
    low_memory=False
)

print(f"Dataset shape: {df.shape}")

Dataset shape: (348146, 103)


## 4. Create Working Copy

A copy of the original dataset is created before any preprocessing.

The original dataset (`df`) is preserved unchanged throughout the notebook, while all preparation steps are applied to `prepared_df`.

In [4]:
prepared_df = df.copy()

print("Working copy created.")
print(prepared_df.shape)

Working copy created.
(348146, 103)


## 5. Duplicate Record Assessment

Duplicate records are evaluated before cleaning.

Previous data understanding showed that repeated INDEX_NR values represent multiple wildlife species involved in the same incident rather than duplicated observations.

Therefore, only complete duplicated rows will be considered for removal.

In [5]:
duplicate_rows = prepared_df.duplicated().sum()

print(f"Complete duplicate rows: {duplicate_rows}")

Complete duplicate rows: 0


### Observation

No fully duplicated records were found in the dataset.

Although repeated `INDEX_NR` values were identified during the Data Understanding phase, they correspond to different wildlife species involved in the same incident rather than duplicated observations.

Therefore, no records were removed at this stage.

## 6. Missing Value Assessment

Missing values are assessed before applying any cleaning strategy.

The evaluation includes:

- the number of missing values;
- the percentage of missing values;
- the distribution of variables across missingness levels;
- the identification of variables requiring individual review.

No variables are removed at this stage. Missingness is first evaluated in relation to each variable's meaning and intended use.

### 6.1 Initial Missing-value Summary

In [6]:
missing_summary = (
    prepared_df
    .isna()
    .sum()
    .to_frame("Missing Count")
)

missing_summary["Missing %"] = (
    missing_summary["Missing Count"]
    / len(prepared_df)
    * 100
).round(2)

missing_summary = (
    missing_summary
    .sort_values("Missing Count", ascending=False)
)

missing_summary.head(20)

,Missing Count,Missing %
INCIDENT_LATITUDE,348146,100.00
INCIDENT_LONGITUDE,348146,100.00
AIRPORT_LATITUDE,348146,100.00
AIRPORT_LONGITUDE,348146,100.00
NR_FATALITIES,348121,99.99
NR_INJURIES,347844,99.91
BIRD_BAND_NUMBER,347352,99.77
EFFECT_OTHER,345363,99.20
ENG_4_POS,344505,98.95
COST_REPAIRS,342721,98.44


The table above shows the variables with the highest missing-value counts and percentages.

Some variables are completely unavailable, while others may only be populated when a specific event or consequence occurs. Therefore, missingness alone is not sufficient to determine whether a variable should be removed.

### 6.2 Observation

In [7]:
def missingness_band(pct):
    if pct == 0:
        return "No missing"
    if pct < 20:
        return "Low (<20%)"
    if pct < 50:
        return "Moderate (20–49.99%)"
    if pct < 80:
        return "High (50–79.99%)"
    return "Very high (>=80%)"


missing_summary["Missingness Band"] = (
    missing_summary["Missing %"]
    .apply(missingness_band)
)

missing_summary["Missingness Band"].value_counts()

Missingness Band
No missing              54
Very high (>=80%)       20
Moderate (20–49.99%)    18
Low (<20%)               6
High (50–79.99%)         5
Name: count, dtype: int64

The dataset contains varying levels of missingness across its 103 variables.

- 54 variables contain no missing values.
- 6 variables have low missingness (<20%).
- 18 variables have moderate missingness (20–49.99%).
- 5 variables have high missingness (50–79.99%).
- 20 variables have very high missingness (≥80%).

Variables with substantial missingness will be evaluated individually rather than removed solely based on a missing-value threshold, because some fields may be structurally missing or populated only under specific operational conditions.

### 6.3 Variables with Very High Missingness


In [8]:
very_high_missing = missing_summary[
    missing_summary["Missing %"] >= 80
]

very_high_missing

,Missing Count,Missing %,Missingness Band
INCIDENT_LATITUDE,348146,100.00,Very high (>=80%)
INCIDENT_LONGITUDE,348146,100.00,Very high (>=80%)
AIRPORT_LATITUDE,348146,100.00,Very high (>=80%)
AIRPORT_LONGITUDE,348146,100.00,Very high (>=80%)
NR_FATALITIES,348121,99.99,Very high (>=80%)
NR_INJURIES,347844,99.91,Very high (>=80%)
BIRD_BAND_NUMBER,347352,99.77,Very high (>=80%)
EFFECT_OTHER,345363,99.20,Very high (>=80%)
ENG_4_POS,344505,98.95,Very high (>=80%)
COST_REPAIRS,342721,98.44,Very high (>=80%)


Variables with at least 80% missing values are reviewed individually.

The purpose is to distinguish between:

- variables that are completely unavailable;
- variables that are sparse by design;
- post-strike consequence variables;
- variables that may still be useful for descriptive analysis;
- variables that are unsuitable for predictive modelling.

### 6.4 Preliminary Assessment

Variables with very high missingness were reviewed individually.

No variables are removed solely because of a high missing-value percentage.

Each variable will be evaluated according to:

- business meaning;
- relevance to predictive modelling;
- potential information leakage;
- availability of meaningful observations.

# 7. Missing Value Strategy

Based on the missing-value assessment, variables are grouped according to their characteristics rather than applying a single rule.

Different handling strategies are required because missing values originate from different causes, including unavailable information, conditional reporting, and post-event outcomes.

The following categories are used to guide data preparation.

In [11]:
missing_strategy = {
    "Drop (100% missing)": [
        "INCIDENT_LATITUDE",
        "INCIDENT_LONGITUDE",
        "AIRPORT_LATITUDE",
        "AIRPORT_LONGITUDE",
    ],

    "Rare-event variables": [
        "NR_FATALITIES",
        "NR_INJURIES",
    ],

    "Post-event variables": [
        "COST_REPAIRS",
        "COST_REPAIRS_INFL_ADJ",
        "COST_OTHER",
        "COST_OTHER_INFL_ADJ",
        "EFFECT",
        "EFFECT_OTHER",
    ],

    "Variables requiring individual review": [
        "ENG_3_POS",
        "ENG_4_POS",
        "ENROUTE_STATE",
        "PRECIPITATION",
        "AOS",
        "LOCATION",
        "OTHER_SPECIFY",
        "BIRD_BAND_NUMBER",
    ]
}

for group, cols in missing_strategy.items():
    print(f"\n{group}")
    print("-" * len(group))
    print(", ".join(cols))


Drop (100% missing)
-------------------
INCIDENT_LATITUDE, INCIDENT_LONGITUDE, AIRPORT_LATITUDE, AIRPORT_LONGITUDE

Rare-event variables
--------------------
NR_FATALITIES, NR_INJURIES

Post-event variables
--------------------
COST_REPAIRS, COST_REPAIRS_INFL_ADJ, COST_OTHER, COST_OTHER_INFL_ADJ, EFFECT, EFFECT_OTHER

Variables requiring individual review
-------------------------------------
ENG_3_POS, ENG_4_POS, ENROUTE_STATE, PRECIPITATION, AOS, LOCATION, OTHER_SPECIFY, BIRD_BAND_NUMBER


### Observation

Variables are grouped according to both their missing-value percentage and their business meaning.

Some variables can be confidently removed (e.g., 100% missing), while others require additional review because they may still contain useful information or represent special operational conditions.

This strategy ensures that preprocessing decisions are based on both data quality and domain knowledge rather than a single missing-value threshold.

# 8. Data Cleaning

## 8.1 Remove Variables with 100% Missing Values

Variables containing no observed values provide no information for descriptive analysis or predictive modelling.

The four variables identified as completely missing are therefore removed from the working dataset.

In [12]:
fully_missing_columns = [
    "INCIDENT_LATITUDE",
    "INCIDENT_LONGITUDE",
    "AIRPORT_LATITUDE",
    "AIRPORT_LONGITUDE",
]

shape_before = prepared_df.shape

prepared_df = prepared_df.drop(
    columns=fully_missing_columns
)

shape_after = prepared_df.shape

print(f"Shape before removal: {shape_before}")
print(f"Shape after removal:  {shape_after}")
print(f"Columns removed: {shape_before[1] - shape_after[1]}")

Shape before removal: (348146, 103)
Shape after removal:  (348146, 99)
Columns removed: 4


In [13]:
remaining_fully_missing = (
    prepared_df
    .isna()
    .all()
    .sum()
)

print(
    "Remaining columns with 100% missing values:",
    remaining_fully_missing
)

Remaining columns with 100% missing values: 0


### Observation

The four variables containing only missing values were successfully removed from the dataset.

After removal, no variables with 100% missing values remain. The remaining missing values will be handled using variable-specific strategies in the subsequent data preparation steps.

## 8.2 Remove Non-informative Identifier Variables

Some variables function only as unique identifiers rather than descriptive features.

These variables provide little or no predictive value and may unnecessarily increase dataset sparsity.

The bird band number is therefore removed from the working dataset.

In [14]:
identifier_columns = [
    "BIRD_BAND_NUMBER"
]

shape_before = prepared_df.shape

prepared_df = prepared_df.drop(
    columns=identifier_columns
)

shape_after = prepared_df.shape

print(f"Shape before removal: {shape_before}")
print(f"Shape after removal:  {shape_after}")
print(f"Columns removed: {shape_before[1] - shape_after[1]}")

Shape before removal: (348146, 99)
Shape after removal:  (348146, 98)
Columns removed: 1


In [15]:
print(
    "BIRD_BAND_NUMBER" in prepared_df.columns
)

False


### Observation

The bird band number was removed because it serves as an identification field rather than a descriptive attribute.

Given its extremely high missing rate and limited analytical value, retaining this variable would not contribute to descriptive analysis or predictive modelling.

## 8.3 Inspect Rare-event Variables

The variables **NR_FATALITIES** and **NR_INJURIES** contain extremely high proportions of missing values.

Before deciding on an imputation strategy, the existing observations are examined to determine whether missing values represent true missing data or simply indicate that no fatalities or injuries occurred.

In [16]:
rare_event_columns = [
    "NR_FATALITIES",
    "NR_INJURIES"
]

prepared_df[rare_event_columns].describe(include="all")

,NR_FATALITIES,NR_INJURIES
count,25.000000,302.000000
mean,2.080000,1.278146
std,1.630951,0.663815
min,1.000000,1.000000
25%,1.000000,1.000000
50%,2.000000,1.000000
75%,2.000000,1.000000
max,8.000000,7.000000


In [17]:
for col in rare_event_columns:
    print(f"\n{col}")
    print(prepared_df[col].value_counts(dropna=False))


NR_FATALITIES
NR_FATALITIES
NaN    348121
1.0        12
2.0         7
3.0         3
8.0         1
5.0         1
4.0         1
Name: count, dtype: int64

NR_INJURIES
NR_INJURIES
NaN    347844
1.0       236
2.0        58
3.0         3
5.0         2
4.0         2
7.0         1
Name: count, dtype: int64


### Decision

The observed values for `NR_FATALITIES` and `NR_INJURIES` begin at 1, and no explicit zero values are present.

This pattern indicates that the fields are populated only when fatalities or injuries are reported. Therefore, missing values are treated as zero rather than as unknown observations.

This is a structural missingness treatment based on the reporting pattern of the variables.

In [18]:
prepared_df[rare_event_columns] = (
    prepared_df[rare_event_columns]
    .fillna(0)
    .astype(int)
)

prepared_df[rare_event_columns].head()

,NR_FATALITIES,NR_INJURIES
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0


In [19]:
rare_event_validation = pd.DataFrame({
    "Missing Count": prepared_df[rare_event_columns].isna().sum(),
    "Minimum Value": prepared_df[rare_event_columns].min(),
    "Maximum Value": prepared_df[rare_event_columns].max(),
    "Data Type": prepared_df[rare_event_columns].dtypes
})

rare_event_validation

,Missing Count,Minimum Value,Maximum Value,Data Type
NR_FATALITIES,0,0,8,int32
NR_INJURIES,0,0,7,int32


### Observation

Missing values in the rare-event variables were replaced with zero.

After treatment:

- both variables contain no missing values;
- `NR_FATALITIES` ranges from 0 to 8;
- `NR_INJURIES` ranges from 0 to 7;
- both variables are stored as integers.

The transformation preserves reported event counts while explicitly representing incidents with no reported fatalities or injuries.

## 8.4 Inspect Post-event Variables

Post-event variables describe consequences or outcomes recorded after a wildlife strike has occurred.

These variables may be useful for descriptive analysis, but they require careful review before predictive modelling because they may directly reveal the outcome being predicted and introduce target leakage.

The following variables are inspected before any cleaning or removal decision is made.

In [20]:
post_event_columns = [
    "COST_REPAIRS",
    "COST_REPAIRS_INFL_ADJ",
    "COST_OTHER",
    "COST_OTHER_INFL_ADJ",
    "EFFECT",
    "EFFECT_OTHER",
]

prepared_df[post_event_columns].describe(include="all").T

,count,unique,top,freq
COST_REPAIRS,5425,1367,$5000.00,210
COST_REPAIRS_INFL_ADJ,5425,3144,$5000.00,14
COST_OTHER,5562,691,$500.00,507
COST_OTHER_INFL_ADJ,5562,1805,$300.00,67
EFFECT,15935,12,Precautionary Landing,9327
EFFECT_OTHER,2783,830,EVASIVE MANEUVER,307


In [21]:
for col in post_event_columns:
    print(f"\n{col}")
    print("-" * len(col))
    print(prepared_df[col].value_counts(dropna=False).head(15))


COST_REPAIRS
------------
COST_REPAIRS
NaN           342721
$5000.00         210
$10000.00        207
$1000.00         179
$500.00          155
$15000.00        154
$20000.00        152
$2000.00         147
$50000.00        137
$30000.00        118
$100000.00       115
$3000.00         105
$25000.00        101
$1500.00          89
$4000.00          87
Name: count, dtype: int64

COST_REPAIRS_INFL_ADJ
---------------------
COST_REPAIRS_INFL_ADJ
NaN          342721
$5000.00         14
$13560.00        13
$13110.00        13
$25160.00        13
$5130.00         12
$1400.00         12
$20685.00        12
$40680.00        12
$3730.00         11
$14920.00        11
$7000.00         11
$18870.00        11
$2800.00         11
$27120.00        10
Name: count, dtype: int64

COST_OTHER
----------
COST_OTHER
NaN          342584
$500.00         507
$100.00         410
$200.00         388
$300.00         371
$1000.00        337
$5000.00        196
$10000.00       170
$2000.00        158
$250.00     

In [22]:
prepared_df[post_event_columns].dtypes

COST_REPAIRS             object
COST_REPAIRS_INFL_ADJ    object
COST_OTHER               object
COST_OTHER_INFL_ADJ      object
EFFECT                   object
EFFECT_OTHER             object
dtype: object

### Decision

The post-event variables contain information recorded after the wildlife strike occurred.

The cost variables describe the financial consequences of the incident, while `EFFECT` and `EFFECT_OTHER` describe operational outcomes such as precautionary landings, aborted take-offs, engine shutdowns, and evasive manoeuvres.

Because the project aims to predict damage severity, these variables may directly reveal the outcome being predicted and introduce target leakage.

Therefore, the six post-event variables are excluded from the predictive feature set. They may still be retained separately for descriptive analysis if required.

In [23]:
post_event_df = prepared_df[
    ["INDEX_NR"] + post_event_columns
].copy()

print("Post-event descriptive dataset shape:", post_event_df.shape)

Post-event descriptive dataset shape: (348146, 7)


### Remove Post-event Variables from Predictive Data

The post-event variables are removed from the working predictive dataset to prevent information leakage.

A separate copy has been retained for possible descriptive analysis.

In [24]:
shape_before = prepared_df.shape

prepared_df = prepared_df.drop(
    columns=post_event_columns
)

shape_after = prepared_df.shape

print(f"Shape before removal: {shape_before}")
print(f"Shape after removal:  {shape_after}")
print(f"Columns removed: {shape_before[1] - shape_after[1]}")

Shape before removal: (348146, 98)
Shape after removal:  (348146, 92)
Columns removed: 6


In [25]:
remaining_post_event_columns = [
    col for col in post_event_columns
    if col in prepared_df.columns
]

print(
    "Post-event columns remaining in predictive dataset:",
    remaining_post_event_columns
)

Post-event columns remaining in predictive dataset: []


# 9. Data Type Assessment

After handling duplicate records, missing values, identifier variables, and post-event variables, the remaining dataset is inspected to verify the data types of all variables.

This assessment helps identify variables that require type conversion before feature engineering and predictive modelling.

In [26]:
prepared_df.dtypes.value_counts()

int64      41
object     37
float64    12
int32       2
Name: count, dtype: int64

In [27]:
prepared_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 348146 entries, 0 to 348145
Data columns (total 92 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   INDEX_NR           348146 non-null  int64  
 1   INCIDENT_DATE      348146 non-null  object 
 2   INCIDENT_MONTH     348146 non-null  int64  
 3   INCIDENT_YEAR      348146 non-null  int64  
 4   TIME               226783 non-null  object 
 5   TIME_OF_DAY        196417 non-null  object 
 6   AIRPORT_ID         348146 non-null  object 
 7   AIRPORT            348146 non-null  object 
 8   RUNWAY             262192 non-null  object 
 9   STATE              301216 non-null  object 
 10  FAAREGION          301216 non-null  object 
 11  LOCATION           45225 non-null   object 
 12  ENROUTE_STATE      6044 non-null    object 
 13  OPID               348143 non-null  object 
 14  OPERATOR           348145 non-null  object 
 15  REG                214611 non-null  object 
 16  FL

### Observation

After the initial data-cleaning stage, the working dataset contains 92 variables.

The dataset consists of:

- 41 integer (`int64`) variables;
- 12 floating-point (`float64`) variables;
- 37 object variables;
- 2 integer (`int32`) variables.

Most numerical variables already use appropriate data types. However, several object variables represent dates or categorical information and require further conversion before feature engineering and predictive modelling.

# 10. Data Type Conversion

## Convert Date Variables

The dataset contains date variables stored as text (`object`).

These variables are converted to datetime format to support chronological analysis and future feature engineering.

In [28]:
prepared_df[
    ["INCIDENT_DATE", "LUPDATE"]
].head()

,INCIDENT_DATE,LUPDATE
0,6/22/1996 0:00:00,12/20/2007 0:00:00
1,6/26/1996 0:00:00,12/20/2007 0:00:00
2,7/1/1996 0:00:00,12/20/2007 0:00:00
3,7/1/1996 0:00:00,12/20/2007 0:00:00
4,7/1/1996 0:00:00,12/20/2007 0:00:00


In [29]:
date_columns = [
    "INCIDENT_DATE",
    "LUPDATE"
]

date_format = "%m/%d/%Y %H:%M:%S"

for col in date_columns:
    prepared_df[col] = pd.to_datetime(
        prepared_df[col],
        format=date_format,
        errors="coerce"
    )

prepared_df[date_columns].dtypes

INCIDENT_DATE    datetime64[ns]
LUPDATE          datetime64[ns]
dtype: object

In [30]:
prepared_df[date_columns].isna().sum()

INCIDENT_DATE    0
LUPDATE          0
dtype: int64

### Observation

The date variables were successfully converted from text to the `datetime` data type.

No conversion errors were detected, indicating that all date values follow a consistent format. These variables are now ready for chronological analysis and future feature engineering.

# 11. Categorical Variable Assessment
## 11.1 Identify Categorical Variables
Most categorical variables are currently stored as the `object` data type.

Before feature engineering, the categorical variables are identified and reviewed to understand their characteristics, cardinality, and suitability for encoding.

In [31]:
categorical_columns = prepared_df.select_dtypes(
    include=["object"]
).columns.tolist()

print(f"Number of categorical variables: {len(categorical_columns)}")

categorical_columns

Number of categorical variables: 35


['TIME',
 'TIME_OF_DAY',
 'AIRPORT_ID',
 'AIRPORT',
 'RUNWAY',
 'STATE',
 'FAAREGION',
 'LOCATION',
 'ENROUTE_STATE',
 'OPID',
 'OPERATOR',
 'REG',
 'FLT',
 'AIRCRAFT',
 'AMA',
 'AMO',
 'AC_CLASS',
 'TYPE_ENG',
 'PHASE_OF_FLIGHT',
 'SKY',
 'PRECIPITATION',
 'DAMAGE_LEVEL',
 'OTHER_SPECIFY',
 'SPECIES_ID',
 'SPECIES',
 'REMARKS',
 'WARNED',
 'NUM_SEEN',
 'NUM_STRUCK',
 'SIZE',
 'COMMENTS',
 'REPORTED_NAME',
 'REPORTED_TITLE',
 'SOURCE',
 'PERSON']

## 11.2 Categorical Cardinality

The number of unique values is calculated for each categorical variable.

Variables with very high cardinality may require different preprocessing strategies from variables containing only a few categories.

In [32]:
categorical_summary = pd.DataFrame({
    "Unique Values": prepared_df[categorical_columns].nunique(dropna=True),
    "Missing Count": prepared_df[categorical_columns].isna().sum()
})

categorical_summary = categorical_summary.sort_values(
    "Unique Values",
    ascending=False
)

categorical_summary

,Unique Values,Missing Count
REMARKS,294362,24921
COMMENTS,212199,118724
REG,47260,133535
FLT,11796,181341
LOCATION,10213,302921
OTHER_SPECIFY,3332,299310
AIRPORT,2779,0
AIRPORT_ID,2779,0
RUNWAY,1560,85954
TIME,1441,121363


## 11.3 High Cardinality Variables

Variables containing a large number of unique categories are identified.

These variables may require special handling because conventional one-hot encoding can substantially increase the dimensionality of the dataset.

In [33]:
high_cardinality = categorical_summary[
    categorical_summary["Unique Values"] > 100
]

high_cardinality

,Unique Values,Missing Count
REMARKS,294362,24921
COMMENTS,212199,118724
REG,47260,133535
FLT,11796,181341
LOCATION,10213,302921
OTHER_SPECIFY,3332,299310
AIRPORT,2779,0
AIRPORT_ID,2779,0
RUNWAY,1560,85954
TIME,1441,121363


### Observation

The categorical variables exhibit varying levels of cardinality.

Several variables (e.g., `REMARKS`, `COMMENTS`, `REG`, `FLT`, and `LOCATION`) contain thousands of unique values and are not suitable for direct one-hot encoding.

Other variables (e.g., `SIZE`, `WARNED`, `SKY`, `NUM_STRUCK`, and `TIME_OF_DAY`) contain only a small number of categories and are suitable candidates for categorical encoding.

Variables with moderate cardinality (e.g., `AIRPORT`, `SPECIES`, `AIRCRAFT`, and `OPERATOR`) will be reviewed individually during feature engineering to determine the most appropriate encoding strategy.

# 12. Remove Non-predictive Variables
## 12.1 Remove Constant Variables

Some variables provide no useful information for predictive modelling.

These variables are removed because they contain only a single unique value across the entire dataset and therefore have no discriminatory power.

In [34]:
constant_columns = [
    "REPORTED_NAME",
    "REPORTED_TITLE"
]

prepared_df[constant_columns].nunique()

REPORTED_NAME     1
REPORTED_TITLE    1
dtype: int64

In [35]:
shape_before = prepared_df.shape

prepared_df = prepared_df.drop(
    columns=constant_columns
)

shape_after = prepared_df.shape

print(f"Shape before removal: {shape_before}")
print(f"Shape after removal:  {shape_after}")
print(f"Columns removed: {shape_before[1]-shape_after[1]}")

Shape before removal: (348146, 92)
Shape after removal:  (348146, 90)
Columns removed: 2


In [36]:
remaining_constant = [
    col for col in constant_columns
    if col in prepared_df.columns
]

print(remaining_constant)

[]


### Observation

The two constant variables were successfully removed.

Since both variables contain only one unique value, they do not contribute any predictive information and would not improve model performance.

## 12.2 Remove Free-text Variables

The variables `COMMENTS` and `REMARKS` contain free-text descriptions.

Natural language processing is outside the scope of this project. Therefore, these variables are excluded from the predictive dataset while structured variables are retained for modelling.

In [37]:
prepared_df[
    ["COMMENTS","REMARKS"]
].head()

,COMMENTS,REMARKS
0,/Legacy Record 100001/,BLOOD ON L FWD NOSE AREA SEEN BY CREW AFTER LDG.
1,/Legacy Record 100002/,CREW SUSPECTED BIRDSTRIKE ON T/O. LOOKED LIKE ...
2,/Legacy Record 100003/,BIRDSTRIKE AT ROTATION. INSPN. NO DMG.
3,/Legacy Record 100004/,"ON FINAL APCH, STRIKE UNDER THE NOSE OF THE CO..."
4,/Legacy Record 100005/,LOUD NOISE WAS HEARD DURING CLIMBOUT THAT SOUN...


In [38]:
text_columns = [
    "COMMENTS",
    "REMARKS"
]

shape_before = prepared_df.shape

prepared_df = prepared_df.drop(
    columns=text_columns
)

shape_after = prepared_df.shape

print(f"Shape before removal: {shape_before}")
print(f"Shape after removal:  {shape_after}")
print(f"Columns removed: {shape_before[1]-shape_after[1]}")

Shape before removal: (348146, 90)
Shape after removal:  (348146, 88)
Columns removed: 2


In [39]:
remaining_text_columns = [
    col for col in text_columns
    if col in prepared_df.columns
]

print(remaining_text_columns)

[]


### Observation

The free-text variables were removed from the predictive dataset.

Although these variables may contain valuable narrative information, processing unstructured text is beyond the scope of this study. Removing them also avoids introducing extremely high-cardinality textual features into the predictive model.

In [40]:
prepared_df[["REG", "FLT"]].head(10)

,REG,FLT
0,NaN,1768
1,NaN,1845
2,NaN,306
3,NaN,510
4,NaN,677
5,N977AA,NaN
6,NaN,484
7,NaN,NaN
8,N690NE,NaN
9,NaN,NaN


## 12.3 Remove High-cardinality Identifier Variables

The variables `REG` and `FLT` represent aircraft registration numbers and flight numbers.

These variables function primarily as identifiers rather than generalizable characteristics of wildlife-strike incidents. They also contain a large number of unique values and substantial missingness.

Including them in the predictive dataset could increase dimensionality and encourage the model to learn incident-specific patterns rather than meaningful relationships. Therefore, they are removed from the working dataset.

In [45]:
print(prepared_df.shape)

(348146, 86)


In [46]:
identifier_columns = [
    "REG",
    "FLT"
]

remaining_identifier_columns = [
    col for col in identifier_columns
    if col in prepared_df.columns
]

print("Identifier columns remaining:", remaining_identifier_columns)

Identifier columns remaining: []


### Observation

The aircraft registration (`REG`) and flight number (`FLT`) variables were successfully removed from the predictive dataset.

These variables function primarily as identifiers and contain extremely high cardinality. Removing them reduces unnecessary model complexity while retaining variables that provide meaningful operational information.

## 12.4 Validate the Working Dataset

The working dataset is validated after removing non-predictive variables to confirm the final dataset dimensions and ensure that all intended variables have been removed successfully.

In [47]:
print("Final dataset shape:")
print(prepared_df.shape)

Final dataset shape:
(348146, 86)


In [48]:
removed_columns = [
    "REPORTED_NAME",
    "REPORTED_TITLE",
    "COMMENTS",
    "REMARKS",
    "REG",
    "FLT"
]

validation = pd.DataFrame({
    "Removed": [
        col not in prepared_df.columns
        for col in removed_columns
    ]
}, index=removed_columns)

validation

,Removed
REPORTED_NAME,True
REPORTED_TITLE,True
COMMENTS,True
REMARKS,True
REG,True
FLT,True


### Observation

All intended non-predictive variables were successfully removed.

The working dataset now contains **348,146 observations** and **86 variables**, providing a cleaner and more suitable feature set for subsequent feature engineering and predictive modelling.

# 13. Export Prepared Dataset

The cleaned and prepared dataset is exported for subsequent exploratory analysis and predictive modelling.

The exported dataset contains only the variables retained after the data preparation process.

In [49]:
prepared_df.to_csv(
    "../data/processed/faa_bird_strikes_prepared.csv",
    index=False
)

print("Prepared dataset saved successfully.")
print(prepared_df.shape)

Prepared dataset saved successfully.
(348146, 86)


# 14. Summary

This notebook completed the data preparation stage for the FAA Wildlife Strike dataset.

The following preprocessing tasks were performed:

- assessed and confirmed the absence of duplicate records;
- evaluated missing values and developed a missing-value treatment strategy;
- removed variables with 100% missing values;
- removed non-informative identifier variables;
- treated rare-event variables (`NR_FATALITIES` and `NR_INJURIES`);
- excluded post-event variables to prevent target leakage;
- converted date variables to the appropriate data type;
- assessed categorical variables and their cardinality;
- removed constant, free-text, and high-cardinality identifier variables;
- exported the prepared dataset for subsequent analysis.

The final prepared dataset contains **348,146 observations** and **86 variables**, providing a clean and consistent dataset for exploratory data analysis and predictive modelling.

In [50]:
print("=" * 60)
print("Data Preparation Completed")
print("=" * 60)

print(f"Final dataset shape : {prepared_df.shape}")
print(f"Rows                : {prepared_df.shape[0]:,}")
print(f"Columns             : {prepared_df.shape[1]}")

print("\nPrepared dataset exported successfully.")
print("Ready for Exploratory Data Analysis (Notebook 03).")

Data Preparation Completed
Final dataset shape : (348146, 86)
Rows                : 348,146
Columns             : 86

Prepared dataset exported successfully.
Ready for Exploratory Data Analysis (Notebook 03).
